In [ ]:
import os
import shutil
import random
from pathlib import Path

# Set random seed for reproducibility
random.seed(42)

# Kaggle paths
BASE_DIR = Path("/kaggle/input/datasets/asfakali2/iruvd-dataset-for-automatic-vehicle-detection/JU_yolov5")
OUTPUT_DIR = Path("/kaggle/working/iruvd_yolov5_processed")

# Source folders
VALID_IMAGES = BASE_DIR / "valid" / "images"
VALID_LABELS = BASE_DIR / "valid" / "labels"

# Destination folders
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "train" / "images").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "train" / "labels").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "val" / "images").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "val" / "labels").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "test" / "images").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "test" / "labels").mkdir(parents=True, exist_ok=True)

# Copy train set
print("Copying train set...")
for src_dir, dest_dir in [
    (BASE_DIR / "train" / "images", OUTPUT_DIR / "train" / "images"),
    (BASE_DIR / "train" / "labels", OUTPUT_DIR / "train" / "labels"),
]:
    for file in src_dir.iterdir():
        shutil.copy2(file, dest_dir / file.name)

# Split valid set into val (50%) and test (50%)
print("Splitting valid set into val/test...")
valid_images = list(VALID_IMAGES.glob("*.jpg")) + list(VALID_IMAGES.glob("*.png"))
random.shuffle(valid_images)

split_idx = int(len(valid_images) * 0.5)
val_files = valid_images[:split_idx]
test_files = valid_images[split_idx:]

def copy_split(files, split_name):
    for img_path in files:
        label_path = VALID_LABELS / img_path.with_suffix(".txt").name
        
        # Copy image
        shutil.copy2(img_path, OUTPUT_DIR / split_name / "images" / img_path.name)
        # Copy label if exists
        if label_path.exists():
            shutil.copy2(label_path, OUTPUT_DIR / split_name / "labels" / label_path.name)

copy_split(val_files, "val")
copy_split(test_files, "test")

print(f"Done! Split: {len(val_files)} val, {len(test_files)} test")

In [ ]:
yaml_content = """path: /kaggle/working/iruvd_yolov5_processed

train: train/images
val: val/images
test: test/images

nc: 14
names:
  0: truck
  1: cyclist
  2: bike
  3: tempo
  4: car
  5: jeep
  6: toto
  7: e-rickshaw
  8: auto-rickshaw
  9: bus
  10: van
  11: cycle-rickshaw
  12: person
  13: taxi
"""

with open("/kaggle/working/iruvd_yolov5_processed/data_corr.yaml", "w") as f:
    f.write(yaml_content)

print("data_corr.yaml saved!")

In [ ]:
from pathlib import Path

processed = Path("/kaggle/working/iruvd_yolov5_processed")
for split in ["train", "val", "test"]:
    imgs = list((processed / split / "images").glob("*.jpg"))
    labels = list((processed / split / "labels").glob("*.txt"))
    print(f"{split}: {len(imgs)} images, {len(labels)} labels")
    
import yaml
with open("/kaggle/working/iruvd_yolov5_processed/data_corr.yaml") as f:
    cfg = yaml.safe_load(f)
print(f"\nClasses: {cfg['nc']}")
print(f"Names: {cfg['names']}")